# 📝 그래프 알고리즘 과제 LV2 정답: 정답 대조, 해상도, 시각화, 경로 (강사용)

각 문제의 **모범답안 + 해설**입니다. 데이터는 **중동 항공 노선망**(130공항 / 11개 국가·지역)입니다.

- 커뮤니티 수와 NMI 는 실행마다 흔들리므로 **범위**로 채점합니다.
- 공항 수·비교 대상 수처럼 결정적인 값만 정확히 채점합니다.
- **3-1 은 그림 재현**(자동 채점 없음), **3-2 는 서술형**입니다.
- **4-2 의 거리 최소·시간 최소 경로는 이 그래프에서 유일**해 경로까지 채점합니다. 편 수 최소 경로는 동점이 여럿이라 **편 수만** 봅니다.

아래 준비 셀들을 먼저 실행하세요. Neo4j 는 반드시 **실습 전용 DB**에 연결합니다.

In [ ]:
# [제공 코드] Neo4j 연결: 이 셀은 실행만 하세요(내용은 이해하지 않아도 됩니다).
# - .env 의 NEO4J_URI/NEO4J_USER/NEO4J_PASSWORD 로 데이터베이스에 연결합니다.
# - run_cypher("쿼리", 파라미터=값) 가 결과를 dict 리스트로 돌려줍니다. 이 헬퍼로 Cypher 를 실행합니다.
# - 반드시 "실습 전용" 데이터베이스에 연결하세요. 아래 실습이 그래프를 지우고 새로 만듭니다.
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase

# 1) 접속 정보 읽기: .env 의 키=값을 환경변수로 올려 둔다(비밀번호를 코드에 적지 않으려고)
load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# os.getenv(키, 기본값): .env 를 못 읽어도 에러가 아니라 이 기본값으로 조용히 넘어간다.
# 그러니 이 셀 마지막 줄에 찍히는 주소가 "실습 전용 DB" 가 맞는지 눈으로 꼭 확인한다
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")

# 2) 드라이버: 노트북과 데이터베이스를 잇는 통로를 하나 열어 둔다(노트북이 끝날 때까지 재사용)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 지금 바로 접속을 시험한다. 여기서 에러가 나면 주소나 계정이 틀린 것


# 3) 공용 헬퍼: 앞으로 모든 Cypher 는 이 함수 하나로 실행한다
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    # session 은 쿼리를 실행하는 단위. with 블록을 벗어나면 알아서 닫힌다
    with driver.session() as session:
        # record.data() 가 한 행을 dict 로 바꾼다. 그 키는 RETURN 에 적은 별칭이 된다
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)

In [ ]:
# [제공 코드] 실습 전용 DB 초기화: 지우기 전에 이 DB 가 맞는지 먼저 확인합니다.
UNIT_LABELS = ['Airport', 'Compound', 'DayType', 'Disease', 'ExDenseEvent', 'ExDisease', 'ExDomestic', 'ExDrug', 'ExEvent', 'ExForeign', 'ExIdKey', 'ExKing', 'ExKinng', 'ExNameKey', 'ExNodeKeyDemo', 'ExPerson', 'ExReign', 'ExScopeEvent', 'ExThrone', 'ExUniqueDemo', 'ExWorld', 'ExYear', 'FlatDrug', 'FlatKing', 'FlatOrder', 'Gene', 'IdKey', 'Line', 'Member', 'NameKey', 'NodeClass', 'NodeDay', 'NodeDrug', 'NodeKing', 'NodeMonth', 'NodeOrder', 'NodeYear', 'Person', 'PharmacologicClass', 'Station', 'SurveyDay', 'SurveyYear', 'Symptom', 'TempStation', 'TryDay', 'TryLine', 'TryStation']   # 이 단원이 만드는 레이블 전부(앞 일차가 남긴 것까지)

# 이 단원 것이 아닌 노드가 하나라도 있으면 지우지 않고 멈춥니다.
# .env 의 주소가 어긋나도 접속은 조용히 성공하므로, 지우기 전에 확인하는 수밖에 없습니다.
# all 로 보는 이유: any 로 보면 :Person:PatientRecord 처럼 한 레이블만 겹치는 남의 노드가 통과합니다
foreign = run_cypher("""
MATCH (n) WHERE size(labels(n)) = 0
   OR NOT all(label IN labels(n) WHERE label IN $unit_labels)
RETURN DISTINCT labels(n) AS labels LIMIT 5""", unit_labels=UNIT_LABELS)
if foreign:
    raise RuntimeError(
        f"{NEO4J_URI} 에 이 단원 것이 아닌 노드가 있습니다: {foreign}\n"
        "다른 실습이나 개인 데이터가 든 DB 로 보여 초기화를 멈췄습니다.\n"
        ".env 의 NEO4J_URI 가 실습 전용 DB 를 가리키는지 먼저 확인하세요.\n"
        "주소가 맞다면 위 레이블은 앞 실습이 남긴 것입니다. UNIT_LABELS 에 더하고 다시 실행하세요.")

# 여기까지 왔으면 이 DB 에는 이 단원이 만든 노드밖에 없습니다.
run_cypher("MATCH (n) DETACH DELETE n")   # DETACH: 노드에 붙은 관계까지 함께 지운다

print("초기화 완료:", NEO4J_URI, "· 남은 노드:", run_cypher("MATCH (n) RETURN count(n) AS n")[0]["n"])

In [ ]:
# [제공 코드] 남아 있는 GDS 투영(메모리 그래프)을 모두 정리: 이 셀은 실행만 하세요.
# 앞 실습의 사본이 남아 있으면 같은 이름으로 다시 투영할 때 충돌합니다.
for row in run_cypher("CALL gds.graph.list() YIELD graphName RETURN graphName"):
    run_cypher("CALL gds.graph.drop($g) YIELD graphName", g=row["graphName"])
print("남은 투영:", run_cypher("CALL gds.graph.list() YIELD graphName RETURN graphName"))

<img src="../images/과제/항공_중동_구성.png" width="900">

| 레이블 | 뜻 | 속성 |
|---|---|---|
| `:Airport` | 공항 130곳 | `iata` 공항 코드 · `name`·`city` 이름과 도시 · `country` 국가·지역 11종 · `lat`·`lon` 좌표 |

| 관계 | 잇는 것 | 속성 |
|---|---|---|
| `ROUTE` | 공항 → 공항 (방향별 978건) | `km` 두 공항 사이 거리 · `hours` km/800 + 1 · `airlines` 운항 항공사 수 |

> **`km`·`hours` 는 원본에 없는 값입니다.** `km` 은 두 공항 좌표로 계산했고, `hours` 는 `km/800 + 1` 로 정했습니다.

In [ ]:
# [제공 코드] 중동 항공 노선망 적재: 이 셀은 실행만 하세요(실습에 쓸 그래프를 만듭니다).
import pandas as pd

airports = pd.read_csv("../data/airports_middle_east_nodes.csv")   # iata, name, city, country, lat, lon
routes = pd.read_csv("../data/airports_middle_east_edges.csv")     # from_iata, to_iata, km, hours, airlines

# iata 로 공항을 찾을 때 전체를 훑지 않도록 유일성 제약을 먼저 겁니다(인덱스가 함께 생깁니다).
run_cypher("CREATE CONSTRAINT airport_iata IF NOT EXISTS FOR (a:Airport) REQUIRE a.iata IS UNIQUE")

# 공항을 노드로 만듭니다.
run_cypher("""UNWIND $rows AS r
    CREATE (:Airport {iata: r.iata, name: r.name, city: r.city, country: r.country,
                      lat: r.lat, lon: r.lon})""", rows=airports.to_dict("records"))

# 노선을 관계로 잇습니다. 왕복이면 CSV 에 두 행이 있어 관계도 둘이 됩니다.
run_cypher("""UNWIND $rows AS r
    MATCH (a:Airport {iata: r.from_iata}), (b:Airport {iata: r.to_iata})
    CREATE (a)-[:ROUTE {km: r.km, hours: r.hours, airlines: r.airlines}]->(b)""",
           rows=routes.to_dict("records"))

print("공항:", run_cypher("MATCH (a:Airport) RETURN count(a) AS n")[0]["n"],
      "· 노선 관계(방향별):", run_cypher("MATCH (:Airport)-[r:ROUTE]->() RETURN count(r) AS n")[0]["n"])

In [ ]:
# [제공 코드] 관계에 hours(소요 시간)를 정의합니다: 이 셀은 실행만 하세요.
# 원본에 '시간'이 없어 우리가 정합니다. 이 상수를 바꾸면 뒤의 최단 시간 경로가 달라집니다.
run_cypher("""MATCH ()-[r:ROUTE]->() SET r.hours = round(r.km / 800.0 + 1.0, 2)""")
print(run_cypher("MATCH ()-[r:ROUTE]->() RETURN min(r.hours) AS shortest, max(r.hours) AS longest, count(*) AS routes"))

In [ ]:
# [제공 코드] 중동 항공 노선망을 무방향으로 투영합니다: 이 셀은 실행만 하세요.
# 기존 air 투영이 있다면 제거합니다.
run_cypher("CALL gds.graph.drop('air', false) YIELD graphName")

# 노선은 방향이 있지만 '어느 공항끼리 이어져 있는가'에는 방향이 의미가 없어 UNDIRECTED 로 만듭니다.
# properties: ['km', 'hours', 'airlines'] 로 관계 속성을 함께 싣습니다. 안 실으면 4-2 에서 거리·시간 최소 경로를 못 잽니다.
stats = run_cypher('''
    CALL gds.graph.project('air', 'Airport',
      {ROUTE: {orientation: 'UNDIRECTED', properties: ['km', 'hours', 'airlines']}})
    YIELD graphName, nodeCount, relationshipCount
    RETURN graphName, nodeCount, relationshipCount''')
print(stats[0])

## 데이터 살펴보기
아래 셀은 **실행만** 하면 됩니다. 규모와 국가·지역 구성을 먼저 확인합니다.

In [ ]:
# [제공 코드] 규모와 국가·지역별 공항 수
import pandas as pd

print('공항 수:', run_cypher('MATCH (a:Airport) RETURN count(a) AS n')[0]['n'])
print('노선 관계 수(방향별):',
      run_cypher('MATCH (:Airport)-[r:ROUTE]->() RETURN count(r) AS n')[0]['n'])
display(pd.DataFrame(run_cypher('''
    MATCH (a:Airport)
    RETURN a.country AS country, count(*) AS airports
    ORDER BY airports DESC''')))

---
# 1. 커뮤니티 탐지와 정답 대조

커뮤니티를 찾아 저장하고, 국가·지역과 맞대 얼마나 맞는지 잽니다(교안_01 2·3절).

## 1-1. 커뮤니티를 찾아 노드에 저장하고 크기 분포 보기
**배경**: 뒤 문제들이 계속 쓰므로 커뮤니티를 먼저 **노드에 저장**해 둡니다.

**요구사항**:
- `gds.leiden.write` 로 `'air'` 투영의 커뮤니티를 **`community`** 속성에 저장하세요.
- 다시 조회해 `iata`·`country`·`community` 세 열의 DataFrame **`found`** 을 만드세요.
- 커뮤니티 개수를 **`n_comm`**, 가장 큰 커뮤니티의 공항 수를 **`max_size`** 에 담으세요.

**예시**: `found` 는 130줄, `n_comm` 은 **한 자리 수**, `max_size` 는 전체의 **3분의 1 남짓**입니다.

<details><summary>힌트</summary>

```text
접근방법:
- write 로 커뮤니티를 그래프에 남긴 뒤, 평범한 Cypher 로 세 열을 읽어 DataFrame 을 만든다.

세부구현:
1. 커뮤니티 탐지를 write 로 호출한다(설정 맵에 저장할 속성 이름을 넣는다).
2. MATCH 로 모든 공항을 찾아 공항 코드·국가·커뮤니티 세 열을 받아 DataFrame 으로 만든다.
3. community 열의 값 종류 수를 n_comm 에, 값별 공항 수의 최댓값을 max_size 에 담는다.
```

</details>

In [ ]:
run_cypher('''
    CALL gds.leiden.write('air', { writeProperty: 'community' })
    YIELD communityCount RETURN communityCount''')

# GDS 가 아니라 평범한 Cypher 로 읽는다
found = pd.DataFrame(run_cypher('''
    MATCH (a:Airport)
    RETURN a.iata AS iata, a.country AS country, a.community AS community'''))
n_comm = found['community'].nunique()
max_size = int(found['community'].value_counts().max())
print('커뮤니티:', n_comm, '개 · 가장 큰 커뮤니티:', max_size, '곳')
print('크기 분포:', sorted(found['community'].value_counts(), reverse=True))

In [ ]:
# [자가채점] 개수, 크기는 범위로, 공항 수는 정확히 확인합니다
assert len(found) == 130, \
    f'found 가 {len(found)}줄입니다. 130곳 전부가 있어야 합니다'
assert list(found.columns) == ['iata', 'country', 'community'], \
    f'열 이름이 {list(found.columns)} 입니다. iata·country·community 세 열이어야 합니다'
assert 2 <= n_comm <= 11, \
    f'커뮤니티가 {n_comm}개로 나왔습니다. air 투영에 Leiden 을 돌렸는지 확인하세요'
assert 20 <= max_size <= 82, \
    f'가장 큰 커뮤니티가 {max_size}곳입니다. 130 에 가깝다면 한 덩어리로 묶인 것입니다'
_stored = run_cypher('MATCH (a:Airport) WHERE a.community IS NOT NULL RETURN count(a) AS n')[0]['n']
assert _stored == 130, \
    'community 속성이 그래프에 저장되지 않았습니다. write 를 썼는지 확인하세요'
print('✅ 통과!')

**해설**: `write` 로 저장해 두면 뒤 문제에서 `a.community` 를 평범한 Cypher 조건으로 쓸 수 있습니다(4-1 이 그렇게 씁니다).

관측에서 커뮤니티는 4~8개, 가장 큰 커뮤니티는 40~52곳이었습니다(새로 적재해 1760회).

## 1-2. 교차표로 커뮤니티와 국가·지역을 맞대 보기
**배경**: 알고리즘은 노선만으로 묶었습니다. 정답과 맞대면 "노선만으로 지도를 얼마나 복원했나"를 알 수 있습니다.

**요구사항**:
- `pd.crosstab` 으로 커뮤니티(행) 대 국가·지역(열) 공항 수 표 **`cross`** 를 만드세요.
- 커뮤니티마다 **순도**(가장 많은 국가·지역이 그 커뮤니티에서 차지하는 비율)를 구해, 커뮤니티 번호를 인덱스로 하는 Series **`purity`** 를 만드세요(소수 셋째 자리 반올림).
- 가장 높은 순도를 **`best_purity`**, 가장 낮은 순도를 **`worst_purity`** 에 담으세요.

**예시**: `cross` 는 몇 행 11열이고, `best_purity` 는 **1.0 에 가깝습니다**(한 나라로만 이뤄진 커뮤니티). `worst_purity` 는 그보다 훨씬 낮습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교차표를 만든 뒤, 행마다 최댓값을 행 합계로 나눈다.

세부구현:
1. pandas 의 교차표 함수에 커뮤니티 열과 국가 열을 넘긴다.
2. 행 방향 합계와 행 방향 최댓값을 각각 구한다(행 방향은 축을 1 로 준다).
3. 최댓값을 합계로 나누고 소수 셋째 자리로 반올림해 purity 에 담는다.
4. purity 의 최댓값과 최솟값을 각각 담는다.
```

</details>

In [ ]:
cross = pd.crosstab(found['community'], found['country'])
# 행 방향(axis=1) 으로 계산해야 '커뮤니티 하나 안에서의 비율'이 된다
purity = (cross.max(axis=1) / cross.sum(axis=1)).round(3)
best_purity = purity.max()
worst_purity = purity.min()
display(cross)
print('커뮤니티별 순도:', dict(purity.sort_values(ascending=False)))
print('가장 높은 순도:', best_purity, '· 가장 낮은 순도:', worst_purity)

In [ ]:
# [자가채점] 표의 모양과 합계는 정확히, 순도는 범위로 확인합니다
assert cross.values.sum() == 130, \
    f'교차표 합계가 {cross.values.sum()} 입니다. 130곳 전부가 들어가야 합니다'
assert cross.shape[1] == 11, \
    f'열이 {cross.shape[1]}개입니다. 국가·지역이 11개이므로 열도 11개여야 합니다'
assert cross.shape[0] == n_comm, \
    '행 수는 커뮤니티 개수와 같아야 합니다. 행과 열을 바꿔 넣지 않았는지 확인하세요'
assert (purity <= 1.0).all() and (purity > 0).all(), \
    '순도는 0 과 1 사이여야 합니다. 최댓값을 합계로 나눴는지 확인하세요'
assert best_purity >= 0.85, \
    f'가장 높은 순도가 {best_purity} 입니다. 행 방향(axis=1)으로 계산했는지 확인하세요'
assert 0.11 <= worst_purity <= 0.8, \
    f'가장 낮은 순도가 {worst_purity} 입니다. 커뮤니티마다 따로 계산했는지 확인하세요'
# 국가별(열 방향)로 구해도 값은 그럴듯하다. 인덱스가 커뮤니티 번호인지로 가른다
assert list(purity.index) == list(cross.index), \
    'purity 의 인덱스가 커뮤니티 번호가 아닙니다. 행 방향(axis=1)으로 나눴는지 확인하세요'
print('✅ 통과!')

**해설**: `axis=1` 이 핵심입니다. 축을 빼먹으면 **열 방향**(한 나라 안에서의 비율)이 되어 전혀 다른 값이 나옵니다.

**읽는 법**: 순도가 1.0 인 커뮤니티는 한 나라의 국내선망이 통째로 잡힌 것이고, 0.31~0.53 언저리인 커뮤니티는 걸프 연안처럼 **여러 나라가 촘촘히 이어진 지역**입니다. 순도가 낮다고 틀린 것이 아니라, 그 지역이 실제로 국경을 넘어 한 덩어리로 오간다는 뜻입니다.

**대안 풀이**: `cross.idxmax(axis=1)` 로 각 커뮤니티의 대표 국가도 함께 뽑아 두면 "3번 커뮤니티는 사실상 튀르키예 국내선" 처럼 이름을 붙여 보고할 수 있습니다.

## 1-3. NMI 로 전체 일치도를 한 숫자로 요약하기
**배경**: 설정을 여러 개 비교하려면 전체 일치도를 한 숫자로 요약해야 편합니다.

**요구사항**:
- `sklearn.metrics` 의 `normalized_mutual_info_score` 로 **국가·지역과 커뮤니티**의 NMI 를 구해 소수 넷째 자리까지 반올림한 값을 **`nmi_leiden`** 에 담으세요.
- **커뮤니티 번호를 통째로 뒤섞은** 라벨과 국가·지역 사이의 NMI 도 구해 **`nmi_shuffled`** 에 담으세요(순서를 섞은 값은 `found['community'].sample(frac=1, random_state=0).values`).

**예시**: 제대로 찾은 커뮤니티의 NMI 는 **섞은 라벨보다 몇 배나 높습니다**. 절대값보다 **벌어지는 폭**을 보세요.

<details><summary>힌트</summary>

```text
접근방법:
- NMI 함수에 정답 라벨과 예측 라벨을 같은 순서로 넘긴다. 섞은 라벨은 비교 기준이다.

세부구현:
1. sklearn 의 해당 함수를 import 한다.
2. 국가 열과 커뮤니티 열을 넘겨 값을 구하고 소수 넷째 자리로 반올림한다.
3. 커뮤니티 열의 순서를 섞은 배열로 한 번 더 구한다(지문의 sample 호출을 그대로 쓴다).
4. 두 값을 출력해 비교한다.
```

</details>

In [ ]:
from sklearn.metrics import normalized_mutual_info_score

nmi_leiden = round(normalized_mutual_info_score(found['country'], found['community']), 4)
# 라벨을 섞으면 '공항과 커뮤니티의 짝'이 끊긴다. 우연히 나올 값의 기준선이다
shuffled = found['community'].sample(frac=1, random_state=0).values
nmi_shuffled = round(normalized_mutual_info_score(found['country'], shuffled), 4)
print('찾은 커뮤니티:', nmi_leiden, '· 섞은 라벨:', nmi_shuffled)

In [ ]:
# [자가채점] NMI 는 실행마다 흔들리므로 범위로 확인합니다
assert 0.56 <= nmi_leiden <= 0.97, \
    f'nmi_leiden 이 {nmi_leiden} 입니다. country 와 community 를 넘겼는지 확인하세요'
assert nmi_shuffled < 0.25, \
    f'nmi_shuffled 가 {nmi_shuffled} 입니다. 라벨을 섞은 배열을 넘겼는지 확인하세요'
assert nmi_leiden > nmi_shuffled * 3, \
    '찾은 커뮤니티의 NMI 가 섞은 라벨보다 훨씬 높아야 합니다. 두 값을 바꿔 담지 않았는지 확인하세요'
# 숫자만 적어도 위 범위는 통과하므로, 1-1 에서 만든 found 로 여기서 직접 다시 잰다.
# 답안이 어떤 방식으로 NMI 를 불렀든 상관없게 채점 셀이 스스로 import 한다
from sklearn.metrics import normalized_mutual_info_score as _nmi
_live_nmi = round(_nmi(found['country'], found['community']), 4)
assert nmi_leiden == _live_nmi, \
    'found 의 두 열로 다시 계산한 값과 다릅니다. 국가 열과 커뮤니티 열을 그대로 넣었는지, 소수 넷째 자리로 반올림했는지 확인하세요'
print('✅ 통과!')

**해설**: 지표를 처음 쓸 때는 **기준선**을 함께 재야 합니다. 값 하나만 보면 높은지 낮은지 알 수 없지만, 아무렇게나 섞었을 때가 얼마인지 알면 뜻이 분명해집니다. 관측에서 Leiden 의 NMI 는 0.714~0.799 였습니다.

**주의**: NMI 는 번호가 아니라 **묶임**을 봅니다. 번호를 전부 +100 해도 값은 그대로지만, **공항과 번호의 짝을 끊으면** 무너집니다.

---
# 2. 알고리즘과 해상도 고르기

다른 알고리즘과 견주고, 해상도 값을 바꿔 봅니다(교안_01 4·5절).

## 2-1. 라벨 전파와 견주어 보기
**배경**: 가장 빠른 라벨 전파를 1-3 과 같은 잣대로 견줍니다. 동점일 때의 규칙이 없어 **그래프를 올린 순서가 답을 바꾼다**는 점을 염두에 두세요.

**요구사항**:
- `gds.labelPropagation.stream` 을 `'air'` 투영에 호출하고, 결과를 `country`·`community` 두 열의 DataFrame **`lpa`** 로 만드세요(국가는 `gds.util.asNode(nodeId).country` 로 함께 받으면 따로 병합하지 않아도 됩니다).
- 커뮤니티 개수를 **`n_lpa`**, 가장 큰 커뮤니티의 공항 수를 **`biggest_lpa`**, 국가·지역 정답과의 NMI 를 소수 넷째 자리까지 반올림해 **`nmi_lpa`** 에 담으세요.

**예시**: 커뮤니티 수는 Leiden 보다 **적게 나오기 쉽고** 그만큼 한 커뮤니티가 큽니다. **새로 적재할 때마다 답이 달라지니** 한 번 돌린 값으로 단정하지 마세요. **개수만 보지 말고 무엇을 더 봐야 하는지 스스로 정해 함께 출력해 보세요**(3-2 에서 묻습니다).

<details><summary>힌트</summary>

```text
접근방법:
- 1-3 과 같은 방식이되 알고리즘 이름만 바꾸고, 개수·최대 크기·NMI 를 구한다.

세부구현:
1. 라벨 전파를 stream 으로 호출해 YIELD 로 노드 id 와 커뮤니티 id 를 받는다.
   - 노드를 되돌려 국가 속성을 함께 받으면 병합 없이 바로 NMI 를 잴 수 있다.
2. DataFrame 으로 만들어 lpa 에 담는다.
3. 커뮤니티 종류 수, 커뮤니티별 공항 수의 최댓값, NMI 를 각각 구해 담는다.
```

</details>

In [ ]:
# country 를 함께 받으면 found 와 병합하지 않아도 바로 NMI 를 잴 수 있다
lpa = pd.DataFrame(run_cypher('''
    CALL gds.labelPropagation.stream('air')
    YIELD nodeId, communityId
    RETURN gds.util.asNode(nodeId).country AS country, communityId AS community'''))
n_lpa = lpa['community'].nunique()
biggest_lpa = int(lpa['community'].value_counts().max())
nmi_lpa = round(normalized_mutual_info_score(lpa['country'], lpa['community']), 4)
print('커뮤니티:', n_lpa, '개 · 가장 큰 커뮤니티:', biggest_lpa, '곳 · NMI:', nmi_lpa)
print('전체 공항 대비 비율:', round(biggest_lpa / 130, 2))
print('1-3 의 Leiden NMI:', nmi_leiden)

In [ ]:
# [자가채점] 값 자체가 적재할 때마다 달라지므로 넓은 범위로만 확인합니다
assert len(lpa) == 130, \
    f'lpa 가 {len(lpa)}줄입니다. 130곳 전부가 있어야 합니다'
assert set(lpa.columns) == {'country', 'community'}, \
    f'열 이름이 {list(lpa.columns)} 입니다. country 와 community 두 열이어야 합니다'
# 적재(내부 번호 순서)마다 답이 갈립니다. 관측 폭은 커뮤니티 1~4개, 최대 커뮤니티 52~130곳, NMI 0.0~0.746 인데,
# 더 벌어질 수 있어 채점은 훨씬 넓게 잡습니다(줄 수만 정확히 봅니다).
assert 1 <= n_lpa <= 6, \
    f'커뮤니티가 {n_lpa}개로 나왔습니다. gds.labelPropagation 을 air 투영에 돌렸는지 확인하세요'
assert 39 <= biggest_lpa <= 130, \
    f'가장 큰 커뮤니티가 {biggest_lpa}곳입니다. 커뮤니티별 공항 수의 최댓값인지 확인하세요'
assert 0.0 <= nmi_lpa <= 0.95, \
    f'nmi_lpa 가 {nmi_lpa} 입니다. country 와 community 를 넘겼는지 확인하세요'
# 라벨 전파의 NMI 폭이 Leiden 과 겹쳐, 대소를 채점하지 않습니다(적재에 따라 뒤집힙니다).
print('✅ 통과!')

**해설**: 라벨 전파는 적재에 따라 커뮤니티 1~4개, 가장 큰 커뮤니티 52~130곳(40%~100%)을 오갑니다. 국가·지역과의 NMI 도 0.0~0.746 로 Leiden 의 0.714~0.799 와 폭이 겹칠 때가 있습니다. 운이 나쁜 적재에서는 거의 전부가 한 덩어리가 되어 **사실상 답을 못 냅니다.**

**왜 이렇게 되나**: 이 지역은 거대한 환승 허브 몇 곳이 나라를 가리지 않고 사방으로 이어져 있어, 다수결로 퍼지는 허브의 라벨이 **국경을 넘어 그대로 번집니다.** 게다가 동점일 때 규칙이 없어 **그래프를 올린 순서가 답을 바꿉니다.**

**실무 지침**: 알고리즘을 고를 때는 개수 하나가 아니라 **최대 커뮤니티 비율·정답과의 일치도·여러 번 돌렸을 때의 폭**을 함께 보십시오.

## 2-2. 해상도를 바꿔 가며 최적값 찾기
**배경**: Leiden 의 `gamma` 는 커뮤니티를 몇 개로 나눌지 정합니다. 정답 라벨이 있으니 어느 값이 좋은지 직접 재 봅니다.

**요구사항**:
- `gamma` 를 `[0.5, 1.0, 1.5, 2.0, 2.5, 3.0]` 로 바꿔 가며 `gds.leiden.stream` 을 돌리세요. 값을 비교하는 것이 목적이므로 **`randomSeed: 42, concurrency: 1`** 을 함께 주세요.
- `gamma`·`communities`·`nmi` 세 열을 가진 DataFrame **`sweep`** 을 만드세요. **행은 `gamma` 가 작은 것부터 순서대로** 담습니다(`nmi` 는 소수 넷째 자리 반올림).
- 커뮤니티 수가 **국가·지역 수(11) 이상이 되는 가장 작은** `gamma` 를 **`gamma_matching`** 에 담으세요.
- **NMI 가 가장 높은** `gamma` 를 **`gamma_best`** 에 담으세요.

**예시**: `gamma` 를 올리면 커뮤니티가 잘게 갈립니다. 두 값이 **같은지 다른지는 재 봐야 알고**, 다르다면 왜 다른지는 3-2 에서 서술합니다.

<details><summary>힌트</summary>

```text
접근방법:
- gamma 목록을 for 문으로 돌면서 매번 커뮤니티 수와 NMI 를 기록해 표로 만든다.

세부구현:
1. 결과를 모을 빈 리스트를 만들고 gamma 목록을 순회한다.
   1-1. 커뮤니티 탐지를 stream 으로 호출하되 설정 맵에 gamma, randomSeed, concurrency 를 넣는다.
        - gamma 는 파라미터로 넘기면 쿼리 문자열을 매번 만들지 않아도 된다.
   1-2. 국가와 커뮤니티를 받아 DataFrame 으로 만든 뒤, 커뮤니티 종류 수와 NMI 를 계산해
        사전으로 리스트에 넣는다.
2. 리스트를 DataFrame 으로 만들어 sweep 에 담는다.
3. communities 가 국가·지역 수 이상인 행만 남기고, 그중 gamma 의 최솟값을 고른다.
4. nmi 가 최대인 행의 gamma 를 고른다(최댓값 행의 인덱스를 찾는 메서드).
```

</details>

In [ ]:
records = []
for gamma in [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]:
    rows = run_cypher('''
        CALL gds.leiden.stream('air', { gamma: $gamma, randomSeed: 42, concurrency: 1 })
        YIELD nodeId, communityId
        RETURN gds.util.asNode(nodeId).country AS country,
               communityId AS community''', gamma=gamma)
    one = pd.DataFrame(rows)
    records.append({'gamma': gamma,
                    'communities': one['community'].nunique(),
                    'nmi': round(normalized_mutual_info_score(one['country'],
                                                              one['community']), 4)})
sweep = pd.DataFrame(records)
display(sweep)

# 커뮤니티 수가 국가·지역 수 이상인 행만 남기고 그중 gamma 가 가장 작은 것을 고른다
gamma_matching = float(sweep.loc[sweep['communities'] >= 11, 'gamma'].min())
gamma_best = float(sweep.loc[sweep['nmi'].idxmax(), 'gamma'])
print('국가 수만큼 갈리는 gamma:', gamma_matching, '· NMI 가 가장 높은 gamma:', gamma_best)

In [ ]:
# [자가채점] 표의 모양과 두 gamma 값을 확인합니다
assert list(sweep.columns) == ['gamma', 'communities', 'nmi'], \
    f'열 이름이 {list(sweep.columns)} 입니다. gamma·communities·nmi 세 열이어야 합니다'
assert len(sweep) == 6, \
    f'sweep 이 {len(sweep)}줄입니다. gamma 6개를 모두 돌렸는지 확인하세요'
assert (sweep.loc[sweep['gamma'].idxmax(), 'communities']
        > sweep.loc[sweep['gamma'].idxmin(), 'communities'] * 2), \
    'gamma 를 올리면 커뮤니티가 훨씬 잘게 갈려야 합니다. gamma 를 설정에 제대로 넘겼는지 확인하세요'
# 스윕만 돌리고 두 값을 손으로 적어도 통과하지 않게, sweep 에서 고른 값인지 확인한다
assert gamma_matching == float(sweep.loc[sweep['communities'] >= 11, 'gamma'].min()), \
    'gamma_matching 은 커뮤니티 수가 국가·지역 수 이상인 행 가운데 gamma 가 가장 작은 값이어야 합니다'
assert gamma_best == float(sweep.loc[sweep['nmi'].idxmax(), 'gamma']), \
    'gamma_best 는 sweep 에서 nmi 가 가장 높은 행의 gamma 여야 합니다'
assert gamma_matching == 2.5, \
    f'gamma_matching 이 {gamma_matching} 입니다. 커뮤니티 수가 11 이상이 되는 가장 작은 gamma 인지 확인하세요'
assert gamma_best in (1.5, 2.0), \
    f'gamma_best 가 {gamma_best} 입니다. nmi 가 최대인 행의 gamma 여야 합니다'
assert gamma_best != gamma_matching, \
    '두 값이 같게 나왔습니다. 국가·지역 수만큼 갈리는 gamma 와 NMI 가 가장 높은 gamma 는 이 데이터에서 다릅니다'
print('✅ 통과!')

**해설**: `gamma` 를 올리면 커뮤니티가 잘게 갈립니다. `randomSeed` 를 고정해 잰 값입니다.

| gamma | 커뮤니티 수 | NMI |
|---|---|---|
| 0.5 | 3 | 0.693~0.716 |
| 1.0 | 4~6 | 0.736~0.778 |
| 1.5 | 5~8 | 0.755~0.852 |
| 2.0 | 8~10 | 0.764~0.865 |
| 2.5 | 11~15 | 0.731~0.775 |
| 3.0 | 16~19 | 0.674~0.726 |

**핵심**: 커뮤니티 수를 정답 라벨 수에 맞춘다고 정답에 가까워지지 않습니다. `gamma=2.5` 라야 커뮤니티 수가 11개 이상이 되지만 NMI 는 그보다 낮은 `gamma=1.5 또는 2.0` 쪽이 더 높습니다. 개수를 맞추려다 **한 나라의 국내선망을 몇 조각으로 쪼개** 버리기 때문입니다.

**해상도가 올라가면 무엇이 보이나**: 낮은 해상도에서는 커뮤니티가 3개뿐이라 걸프 연안이 한 덩어리로 뭉치고, 높은 해상도에서는 16~19개까지 갈려 나라 안의 지역 노선망까지 떨어집니다. 어느 쪽이 옳은가는 **무엇을 보려는가**에 달려 있습니다.

**왜 `randomSeed` 를 고정했나**: 실행마다 결과가 흔들리면 차이가 gamma 때문인지 우연 때문인지 가릴 수 없습니다. 다만 이 값은 **한 번 적재한 그래프 위에서만** 결과를 고정하므로, 보고할 때는 표를 그대로 옮기지 말고 경향과 함께 적으십시오.

아래 폰트 셀을 먼저 실행하세요. 3-1 그림의 한글 라벨이 이 설정을 씁니다.

In [ ]:
# [제공 코드] 한글 폰트: 이 셀은 실행만 하세요.
import platform

import matplotlib.pyplot as plt

# 한글 폰트: 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == "Windows":
    KOREAN_FONT = "Malgun Gothic"
elif platform.system() == "Darwin":          # macOS
    KOREAN_FONT = "AppleGothic"
else:                                        # Linux (Colab 등)
    KOREAN_FONT = "NanumGothic"

plt.rcParams["font.family"] = KOREAN_FONT
plt.rcParams["axes.unicode_minus"] = False   # 마이너스(-) 부호 깨짐 방지

---
# 3. 시각화와 보고

결과를 그림으로 만들고, 무엇을 함께 보고해야 하는지 씁니다(교안_01 6절. 누적 막대는 7일차 시각화 교안에서 다뤘습니다).

## 3-1. 커뮤니티별 국가·지역 구성을 누적 막대로 그리기
**배경**: **누적 막대**로 그리면 어느 커뮤니티가 한 나라로만 이뤄졌고 어느 커뮤니티가 섞였는지 보입니다.

<img src="../images/과제/커뮤니티_국가구성.png" width="720">

**요구사항**: 위 그림처럼 그리세요(자동 채점은 없습니다).

- 가로축은 커뮤니티, 세로축은 공항 수, **색은 국가·지역**입니다.
- 한 커뮤니티의 막대 안에 그 커뮤니티의 국가·지역 구성이 **쌓여** 보여야 합니다.
- 커뮤니티는 **공항이 많은 순서**로 왼쪽부터 세웁니다.
- 범례에 국가·지역 이름을 넣고, 제목과 축 라벨을 한글로 답니다(`fontfamily=KOREAN_FONT` 를 잊지 마세요).

> 커뮤니티 번호와 색 순서는 실행마다 달라집니다. **모양**만 같으면 됩니다.

**확인 기준**: 막대 개수는 `n_comm` 과 같고, 막대 하나의 **쌓아 올린 전체 높이**는 그 커뮤니티의 공항 수이며, 범례에는 국가·지역 11개가 모두 나옵니다.

<details><summary>힌트</summary>

```text
접근방법:
- 1-2 의 교차표를 공항 수 순으로 정렬한 뒤, 국가마다 막대를 앞선 막대 위에 쌓아 올린다.

세부구현:
1. 교차표의 행 합계로 정렬해 공항이 많은 커뮤니티가 앞에 오게 한다.
2. 새 그림과 축을 만든다(앞 그림에 겹쳐 그리지 않도록).
3. 국가 열을 하나씩 돌면서 막대를 그린다.
   3-1. 쌓아 올리려면 지금까지 그린 높이의 합을 아래쪽 시작점으로 준다.
   3-2. 각 막대에 국가 이름을 라벨로 붙여 범례에 나오게 한다.
4. 축 눈금에 커뮤니티 번호를 적고, 제목·축 라벨·범례를 단다.

대안 풀이:
- 정렬한 교차표에 plot(kind='bar', stacked=True, ax=ax) 를 부르면 pandas 가 쌓기와
  범례를 알아서 해 준다. 색과 글꼴을 직접 다루려면 위 방법이 낫다.
```

</details>

In [ ]:
import numpy as np

# 공항이 많은 커뮤니티가 왼쪽에 오게 정렬한다
ordered = cross.loc[cross.sum(axis=1).sort_values(ascending=False).index]
# 국가·지역이 열 개가 넘어 넉넉한 팔레트를 쓴다
PALETTE = plt.get_cmap('tab20').colors

fig, ax = plt.subplots(figsize=(10, 5))
positions = np.arange(len(ordered))
# 쌓아 올리려면 '지금까지 그린 높이'를 계속 들고 다녀야 한다
bottom = np.zeros(len(ordered))
for i, country in enumerate(ordered.columns):
    heights = ordered[country].values
    ax.bar(positions, heights, bottom=bottom, color=PALETTE[i % len(PALETTE)],
           label=country)
    bottom = bottom + heights
ax.set_xticks(positions)
ax.set_xticklabels([str(c) for c in ordered.index])
ax.set_xlabel('커뮤니티 번호', fontfamily=KOREAN_FONT)
ax.set_ylabel('공항 수', fontfamily=KOREAN_FONT)
ax.set_title('커뮤니티별 국가·지역 구성', fontfamily=KOREAN_FONT)
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

**해설**: 누적 막대는 `bottom` 인자가 전부입니다. **앞서 그린 높이의 합**을 `bottom` 으로 넘겨야 위로 쌓이고, 안 넘기면 막대가 서로 겹쳐 가려집니다.

**읽는 법**: 한 색으로만 채워진 막대는 한 나라의 국내선망, 여러 색이 섞인 막대는 국경을 가로지르는 노선망입니다. 4-1 에서 그 안의 공항들을 찾아냅니다.

**대안 풀이**: `ordered.plot(kind='bar', stacked=True, ax=ax)` 한 줄로도 같은 그림이 나옵니다. 다만 색과 범례를 직접 다루려면 위처럼 그리는 편이 낫습니다.

## 3-2. (서술형) 이 결과를 어떻게 보고할 것인가
**배경**: 실무에서는 숫자를 구하는 것보다 **어떻게 보고하느냐**가 더 자주 문제가 됩니다.

**요구사항**: 아래 세 가지를 **4~6문장**으로 서술하세요(아래 markdown 셀에 작성. 쓴 뒤 정답 노트북의 모범 서술과 견주어 보세요).

1. 2-2 에서 `gamma_matching` 과 `gamma_best` 가 달랐습니다. **커뮤니티 수를 정답 라벨 수에 맞추는 것이 왜 좋은 목표가 아닌지** 설명하세요.
2. 2-1 의 라벨 전파 결과를 "커뮤니티를 몇 개 찾았다"고만 보고하면 무엇을 놓치는지, **함께 보고해야 할 숫자**가 무엇인지 쓰세요.
3. 동료가 "NMI 가 1.0 이 아니니 알고리즘이 실패한 것"이라고 말합니다. **어떻게 반박하겠습니까?**

**모범 서술**

커뮤니티 수를 정답 라벨 수에 맞추는 것은 겉모습만 맞추는 일입니다. `gamma=2.5` 라야 커뮤니티 수가 국가·지역 수 이상이 되지만 NMI 는 그보다 낮은 `gamma=1.5 또는 2.0` 쪽이 더 높았습니다. 개수를 맞추려고 해상도를 올리면 한 나라의 국내선망이 몇 조각으로 쪼개져, 개수는 비슷해져도 묶임은 정답에서 멀어지기 때문입니다.

라벨 전파를 "커뮤니티 몇 개"로만 보고하면 **가장 큰 커뮤니티의 비율**(적재에 따라 40%~100%)과 **정답과의 일치도**(NMI 0.0~0.746 로 Leiden 의 0.714~0.799 에 못 미침)가 가려집니다. 개수와 함께 최대 커뮤니티의 공항 수·비율, NMI, 여러 번 돌렸을 때의 폭을 적어야 하고, 이 경우에는 "라벨 전파를 쓰지 않는 편이 낫다"는 판단까지 붙여야 합니다.

NMI 가 1.0 이 아닌 것은 실패가 아닙니다. 나라 경계와 실제 항공 노선이 같을 이유가 없기 때문입니다. 어긋난 부분은 오류가 아니라 **국경을 가로지르는 노선망**일 가능성이 크고, 그것을 찾아내는 것이 이 분석의 목적입니다. 오히려 NMI 가 1.0 이라면 "국가 목록을 그대로 베낀 결과"라고 말해야 합니다.

---
# 4. 노드 유사도와 최단 경로

닮은 공항을 찾고, 무엇을 최소화하느냐에 따라 길이 달라지는지 봅니다(교안_02 1·2·3절).

## 4-1. 국경을 가로지르는 닮은 공항 찾기
**배경**: 3-1 그림에서 여러 색이 섞인 막대 안의 공항, 곧 **같은 커뮤니티인데 국가·지역이 다른** 짝을 뽑아냅니다.

**요구사항**:
- `gds.nodeSimilarity.stream` 을 `'air'` 투영에 `topK: 5, degreeCutoff: 10` 으로 호출하세요.
- **커뮤니티는 같고 국가·지역은 다른** 짝만 남기고, 코드가 **작은 쪽을 `airport_a`** 로 두어 같은 짝이 한 번만 나오게 하세요.
- `airport_a`·`country_a`·`airport_b`·`country_b`·`similarity` 다섯 열의 DataFrame **`bridges`** 를 유사도 내림차순으로 만드세요(`similarity` 는 소수 넷째 자리 반올림).
- 1위 짝의 유사도를 **`top_similarity`** 에 담으세요.

**예시**: 짝이 여럿 나오고 1위 유사도는 **절반을 넘습니다**. 어느 짝이 1위인지는 그 실행의 커뮤니티 결과에 따라 달라집니다.

<details><summary>힌트</summary>

```text
접근방법:
- 유사도 결과에서 노드를 되돌린 뒤, 커뮤니티가 같고 국가가 다른 행만 남긴다.
- 같은 짝이 양쪽 방향으로 두 번 나오므로 공항 코드 크기 비교로 한쪽만 남긴다.

세부구현:
1. 노드 유사도를 stream 으로 호출하고 설정 맵에 상위 개수와 최소 이웃 수를 넣는다.
2. WITH 로 두 노드 id 를 노드로 되돌린다.
3. WHERE 로 세 조건을 건다.
   3-1. 한쪽 공항 코드가 다른 쪽보다 작은 행만 남긴다(중복 제거).
   3-2. 두 공항의 커뮤니티 속성이 같다.
   3-3. 두 공항의 국가·지역이 다르다.
4. 다섯 열을 유사도 내림차순으로 받아 DataFrame 으로 만들고, 첫 행의 유사도를 담는다.
```

</details>

In [ ]:
# 세 조건(중복 제거, 같은 커뮤니티, 다른 국가)을 Cypher 쪽에서 한 번에 건다
bridges = pd.DataFrame(run_cypher('''
    CALL gds.nodeSimilarity.stream('air', { topK: 5, degreeCutoff: 10 })
    YIELD node1, node2, similarity
    WITH gds.util.asNode(node1) AS a, gds.util.asNode(node2) AS b, similarity
    // 같은 짝이 양쪽으로 두 번 나오므로 한쪽만 남긴다
    WHERE a.iata < b.iata
      AND a.community = b.community
      AND a.country <> b.country
    RETURN a.iata AS airport_a, a.country AS country_a,
           b.iata AS airport_b, b.country AS country_b,
           round(similarity, 4) AS similarity
    ORDER BY similarity DESC'''))
top_similarity = bridges['similarity'].iloc[0]
display(bridges.head(8))
print('1위 유사도:', top_similarity)

In [ ]:
# [자가채점] 조건이 실제로 걸렸는지 DB 로 다시 확인합니다
assert list(bridges.columns) == ['airport_a', 'country_a', 'airport_b', 'country_b',
                                 'similarity'], \
    f'열 이름이 {list(bridges.columns)} 입니다. 지문의 다섯 열 이름을 그대로 쓰세요'
assert 1 <= len(bridges) <= 80, \
    f'짝이 {len(bridges)}개 나왔습니다. 세 조건(중복 제거·같은 커뮤니티·다른 국가)을 모두 걸었는지 확인하세요'
assert (bridges['country_a'] != bridges['country_b']).all(), \
    '국가·지역이 같은 짝이 섞여 있습니다. a.country <> b.country 조건을 확인하세요'
assert (bridges['airport_a'] < bridges['airport_b']).all(), \
    'airport_a 가 airport_b 보다 코드가 작아야 합니다. 양쪽 방향 가운데 한쪽만 남기세요'
_first = bridges.iloc[0]
_same = run_cypher('''
    MATCH (a:Airport { iata: $a }), (b:Airport { iata: $b })
    RETURN a.community = b.community AS same''',
    a=_first['airport_a'], b=_first['airport_b'])[0]['same']
assert _same, '1위 짝이 같은 커뮤니티가 아닙니다. a.community = b.community 조건을 확인하세요'
assert list(bridges['similarity']) == sorted(bridges['similarity'], reverse=True), \
    'bridges 가 유사도 내림차순이 아닙니다. ORDER BY 를 확인하세요'
assert top_similarity == bridges['similarity'].max(), \
    'top_similarity 는 bridges 의 1위 유사도여야 합니다'
assert 0.43 <= top_similarity <= 0.88, \
    f'1위 유사도가 {top_similarity} 입니다. degreeCutoff 를 10 으로 줬는지, 국가 조건이 빠지지 않았는지 확인하세요'
print('✅ 통과!')

**해설**: 세 조건을 동시에 걸어야 합니다. 하나라도 빠지면 다른 명단이 나옵니다.

- `a.iata < b.iata` 를 빼면 같은 짝이 두 줄로 나옵니다.
- `a.community = b.community` 를 빼면 "국가가 다른데 닮은 공항"이 되어 노선망이 전혀 다른 곳까지 섞입니다.
- `degreeCutoff` 를 빼면 노선이 한두 곳뿐인 작은 공항들이 유사도 1.0 으로 상위를 채웁니다(이 그래프에서 cutoff 10 을 걸면 비교 대상이 27곳으로 줄어듭니다).

**1위 짝**: 관측에서는 쿠웨이트(KWI) 와 도하(DOH) 가 0.6774 로 늘 1위였습니다(1760회 모두 같은 커뮤니티). 나라는 다르지만 걸프 연안의 같은 노선망을 써서 가는 곳이 거의 같습니다. **국경이 아니라 노선이 이웃을 정합니다.**

## 4-2. 무엇을 최소화하느냐에 따라 길이 달라지는지 보기
**배경**: 갈아타기를 줄이는 길과 거리를 줄이는 길은 다를 수 있습니다. 준비 셀에서 새긴 `hours` 까지 더한 **세 잣대**로 같은 두 공항을 재 봅니다.

**요구사항**:
- `gds.shortestPath.dijkstra.stream` 으로 **두바이(DXB) 에서 반(VAN)** 까지의 최단 경로를 **세 번** 구하세요.
- 가중치를 주지 않고 구한 편 수를 정수로 **`hops`** 에 담으세요.
- `relationshipWeightProperty` 에 `'km'` 을 주고 구해, 총거리를 **정수로 반올림**해 **`km_total`**, 그 경로의 편 수를 정수로 **`km_hops`**, 지나는 공항 **코드 목록**(출발·도착 포함, 파이썬 리스트)을 **`km_route`** 에 담으세요.
- `'hours'` 로도 같은 세 가지를 구해 **`hours_total`**(소수 둘째 자리 반올림)·**`hours_hops`**·**`hours_route`** 에 담으세요.
- 편 수는 `nodeIds` 의 길이에서 1 을 뺀 값입니다(`size(nodeIds) - 1`).
- 세 결과를 한 줄씩 출력해 견주세요.

**예시**: `km_route` 와 `hours_route` 는 `['DXB', ..., 'VAN']` 모양의 공항 코드 리스트입니다. 세 잣대가 **같은 길을 고르는지, 편 수가 늘어나는지 줄어드는지**를 직접 확인하세요.

<details><summary>힌트</summary>

```text
접근방법:
- 세 호출의 차이는 설정 맵에 가중치 속성 한 줄이 있느냐, 무엇을 넣느냐뿐이다.
  함수 하나로 만들어 세 번 부른다.

세부구현:
1. MATCH 로 두 공항 노드를 코드로 찾는다(번호가 아니라 노드 자체를 넘긴다).
2. 최단 경로를 stream 으로 호출해 총비용과 노드 목록을 함께 받는다.
   - 노드 목록은 내부 id 라 gds.util.asNode 로 되돌려 공항 코드를 꺼낸다.
3. 가중치 없이 한 번 호출해 편 수를 정수로 hops 에 담는다.
4. 설정 맵에 가중치 속성 이름을 넣어 거리로 한 번, 시간으로 한 번 더 호출한다.
   4-1. 총비용을 각각 반올림해 담는다(거리는 정수, 시간은 소수 둘째 자리).
   4-2. 노드 목록의 길이에서 1 을 뺀 값을 편 수로, 공항 코드 목록을 그대로 담는다.
```

</details>

In [ ]:
def route(a, b, weight=None):
    """두 공항 사이의 최단 경로를 (총값, 편 수, 공항 코드 목록) 으로 돌려준다."""
    # 가중치를 안 쓸 때는 relationshipWeightProperty 키를 아예 넣지 않는다
    weight_line = '' if weight is None else f", relationshipWeightProperty: '{weight}'"
    rows = run_cypher(f'''
        MATCH (x:Airport {{ iata: $a }}), (y:Airport {{ iata: $b }})
        CALL gds.shortestPath.dijkstra.stream('air',
             {{ sourceNode: x, targetNode: y{weight_line} }})
        YIELD totalCost, nodeIds
        RETURN totalCost AS total, size(nodeIds) - 1 AS hops,
               [n IN nodeIds | gds.util.asNode(n).iata] AS airports''', a=a, b=b)
    return rows[0]['total'], rows[0]['hops'], rows[0]['airports']


# 가중치를 안 주면 총비용이 곧 편 수다
hops = route('DXB', 'VAN')[1]
_km_total, km_hops, km_route = route('DXB', 'VAN', 'km')
_hours_total, hours_hops, hours_route = route('DXB', 'VAN', 'hours')
# totalCost 는 실수로 온다. 거리는 반올림, 시간은 소수 둘째 자리로 둔다
km_total = round(_km_total)
hours_total = round(_hours_total, 2)
print(f'편 수 최소: {hops}편')
print(f'거리 최소: {km_total}km ({km_hops}편) {km_route}')
print(f'시간 최소: {hours_total}시간 ({hours_hops}편) {hours_route}')

In [ ]:
# [자가채점] 편 수 최소 경로는 동점이 여럿이라 편 수만 보고,
# 거리 최소, 시간 최소 경로는 이 그래프에서 유일하므로 경로까지 대조합니다
assert isinstance(hops, int) and isinstance(km_hops, int) and isinstance(hours_hops, int), \
    '편 수는 정수로 담으세요(size(nodeIds) - 1 은 정수로 옵니다)'
assert hops == 2, \
    f'hops 가 {hops} 입니다. 가중치를 주지 말고 DXB 에서 VAN 까지 쟀는지 확인하세요'
assert km_hops == 3 and km_total == 3677, \
    f'거리 최소가 {km_total}km({km_hops}편) 입니다. relationshipWeightProperty 에 km 을 넘겼는지, 총거리를 정수로 반올림했는지 확인하세요'
assert list(km_route) == ['DXB', 'DOH', 'ESB', 'VAN'], \
    f'거리 최소 경로가 {list(km_route)} 입니다. 이 그래프에서 거리 최소 경로는 하나뿐입니다'
assert hours_hops == 2 and hours_total == 7.25, \
    f'시간 최소가 {hours_total}시간({hours_hops}편) 입니다. hours 를 가중치로 넘겼는지, 소수 둘째 자리로 반올림했는지 확인하세요'
assert list(hours_route) == ['DXB', 'SAW', 'VAN'], \
    f'시간 최소 경로가 {list(hours_route)} 입니다. 이 그래프에서 시간 최소 경로는 하나뿐입니다'


# 두 경로가 실제로 이어져 있고 합이 맞는지 DB 로 되짚는다.
# 왕복 노선은 관계가 둘이라 min 으로 한 개만 센다
def _walk(ids, prop):
    return run_cypher(f'''
        UNWIND range(0, size($ids) - 2) AS step
        MATCH (x:Airport {{ iata: $ids[step] }})
              -[r:ROUTE]-(y:Airport {{ iata: $ids[step + 1] }})
        WITH step, min(r.{prop}) AS leg
        RETURN sum(leg) AS total, count(*) AS steps''', ids=list(ids))[0]


_km_walk = _walk(km_route, 'km')
assert _km_walk['steps'] == km_hops and round(_km_walk['total']) == km_total, \
    f"km_route 를 따라가 보니 {_km_walk['steps']}구간 {_km_walk['total']}km 입니다. " \
    '같은 경로에서 나온 값이어야 합니다'
_hours_walk = _walk(hours_route, 'hours')
assert _hours_walk['steps'] == hours_hops and round(_hours_walk['total'], 2) == hours_total, \
    f"hours_route 를 따라가 보니 {_hours_walk['steps']}구간 " \
    f"{round(_hours_walk['total'], 2)}시간입니다. 같은 경로에서 나온 값이어야 합니다"
print('✅ 통과!')

**해설**: 세 호출의 차이는 `relationshipWeightProperty` 한 줄뿐인데 그 한 줄이 답을 바꿉니다.

| 잣대 | 편 수 | 경로 |
|---|---|---|
| 갈아타기 최소 | 2편 | 같은 길이가 2개라 엔진이 그중 하나를 준다 |
| 거리 최소 | 3편 | 두바이(DXB) → 도하(DOH) → 앙카라(ESB) → 반(VAN) (3677km) |
| 시간 최소 | 2편 | 두바이(DXB) → 이스탄불 사비하괵첸(SAW) → 반(VAN) (7.25시간) |

**읽는 법**: 거리만 줄이려 하면 한 번 더 갈아타게 됩니다(2편이 3편으로). 거리 계산에는 **갈아타는 값이 들어 있지 않아** 짧은 구간을 이어 붙이는 쪽이 이기기 때문입니다. 반대로 `hours` 에는 구간마다 1시간이 얹혀 있어 갈아탈수록 손해라, 시간 최소는 다시 2편으로 돌아옵니다.

**무엇을 최소화할지는 우리가 정합니다.** `hours` 의 800km/h 와 1시간은 원본 데이터에 있는 값이 아니라 **우리가 세운 가정**입니다. 환승을 2시간으로 잡으면 답이 또 달라지므로 보고서에는 반드시 어떻게 정의했는지 적습니다.

**흔한 실수**

- 투영할 때 `properties: ['km', 'hours', 'airlines']` 를 빠뜨리면 `Relationship weight property 'km' not found` 가 납니다. 알고리즘이 아니라 **투영**을 고쳐야 합니다(서두의 투영 셀을 다시 실행하세요).
- 가중치를 안 넘기면 총비용이 편 수로 나옵니다. 값이 한 자리 수라면 그것입니다.
- `hours` 를 투영보다 **뒤에** 새기면 투영에 안 실립니다. 투영은 그 시점의 스냅샷이기 때문입니다.

---
수고했어요! 커뮤니티를 **정답 라벨과 대조하고**(교차표·순도·NMI), 해상도를 조절하고, 결과를 **그림과 문장으로 설명**하고, **무엇을 최소화하느냐에 따라 최단 경로가 달라진다**는 것까지 확인했습니다. 36일차에서는 이 그래프를 언어 모델과 잇는 지식그래프로 넘어갑니다.